In [36]:
# Part 3: Single-View Geometry

## Usage
# This code snippet provides an overall code structure and some interactive plot interfaces for the *Single-View Geometry* section of Assignment 3. In [main function](#Main-function), we outline the required functionalities step by step. Some of the functions which involves interactive plots are already provided, but [the rest](#Your-implementation) are left for you to implement.

## Package installation
# - In this code, we use `tkinter` package. Installation instruction can be found [here](https://anaconda.org/anaconda/tk).

In [37]:
# Common imports

In [38]:
import matplotlib
matplotlib.use("TkAgg")   # Here we are setting the backend for matplotlib to use TkAgg.
import matplotlib.pyplot as plt
plt.ion()   # Here we are sets=ting the interactive mode of the Matplotlib library, allowing for real-time updates to plots in a figure.
import numpy as np
from PIL import Image

In [39]:
# Here we are defining the variable zero.
zero = 0

# Here we are defining the variable one.
one = 1

# Here we are defining the variable two.
two = 2

# Here we are defining the variable three.
three = 3

# Here we are defining the variable pointFive.
pointFive = 0.5

# Here we are defining the variable inchesNumber.
inchesNumber = 66

In [40]:
# Provided functions

In [41]:
def get_input_lines(im, min_lines=3):
    """
    Allows user to input line segments; computes centers and directions.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        min_lines: minimum number of lines required
    Returns:
        n: number of lines from input
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        centers: np.ndarray of shape (3, n)
            where each column denotes the homogeneous coordinates of the centers
    """
    n = 0
    lines = np.zeros((3, 0))
    centers = np.zeros((3, 0))

    plt.figure()
    plt.imshow(im)
    plt.show()
    print('Set at least %d lines to compute vanishing point' % min_lines)
    while True:
        print('Click the two endpoints, use the right key to undo, and use the middle key to stop input')
        clicked = plt.ginput(2, timeout=0, show_clicks=True)
        if not clicked or len(clicked) < 2:
            if n < min_lines:
                print('Need at least %d lines, you have %d now' % (min_lines, n))
                continue
            else:
                # Stop getting lines if number of lines is enough
                break

        # Unpack user inputs and save as homogeneous coordinates
        pt1 = np.array([clicked[0][0], clicked[0][1], 1])
        pt2 = np.array([clicked[1][0], clicked[1][1], 1])
        # Get line equation using cross product
        # Line equation: line[0] * x + line[1] * y + line[2] = 0
        line = np.cross(pt1, pt2)
        lines = np.append(lines, line.reshape((3, 1)), axis=1)
        # Get center coordinate of the line segment
        center = (pt1 + pt2) / 2
        centers = np.append(centers, center.reshape((3, 1)), axis=1)

        # Plot line segment
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color='b')

        n += 1

    return n, lines, centers

def plot_lines_and_vp(im, lines, vp):
    """
    Plots user-input lines and the calculated vanishing point.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        vp: np.ndarray of shape (3, )
    """
    bx1 = min(1, vp[0] / vp[2]) - 10
    bx2 = max(im.shape[1], vp[0] / vp[2]) + 10
    by1 = min(1, vp[1] / vp[2]) - 10
    by2 = max(im.shape[0], vp[1] / vp[2]) + 10

    plt.figure()
    plt.imshow(im)
    for i in range(lines.shape[1]):
        if lines[0, i] < lines[1, i]:
            pt1 = np.cross(np.array([1, 0, -bx1]), lines[:, i])
            pt2 = np.cross(np.array([1, 0, -bx2]), lines[:, i])
        else:
            pt1 = np.cross(np.array([0, 1, -by1]), lines[:, i])
            pt2 = np.cross(np.array([0, 1, -by2]), lines[:, i])
        pt1 = pt1 / pt1[2]
        pt2 = pt2 / pt2[2]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], 'g')

    plt.plot(vp[0] / vp[2], vp[1] / vp[2], 'ro')
    plt.show()

def get_top_and_bottom_coordinates(im, obj):
    """
    For a specific object, prompts user to record the top coordinate and the bottom coordinate in the image.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        obj: string, object name
    Returns:
        coord: np.ndarray of shape (3, 2)
            where coord[:, 0] is the homogeneous coordinate of the top of the object and coord[:, 1] is the homogeneous
            coordinate of the bottom
    """
    plt.figure()
    plt.imshow(im)

    print('Click on the top coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x1, y1 = clicked[0]
    # Uncomment this line to enable a vertical line to help align the two coordinates
    # plt.plot([x1, x1], [0, im.shape[0]], 'b')
    print('Click on the bottom coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x2, y2 = clicked[0]

    plt.plot([x1, x2], [y1, y2], 'b')

    return np.array([[x1, x2], [y1, y2], [1, 1]])

In [42]:
# Your implementation

In [43]:
# Here we are creating a function called get_vanishing_point.
def get_vanishing_point(p1):
    """
    Solves for the vanishing point using the user-input lines.
    """

    # Here we are retrieving the eigenvector corresponding to the minimum eigenvalue of p1.dot(p1.T).
    return (np.linalg.eig(p1.dot(p1.T))[one][:, np.argmin(np.linalg.eig(p1.dot(p1.T))[zero])] / np.linalg.eig(p1.dot(p1.T))[one][:, np.argmin(np.linalg.eig(p1.dot(p1.T))[zero])][-one])

# Here we are creating a function called get_horizon_line.
def get_horizon_line(p1):
    """
    Calculates the ground horizon line.
    """

    # Here we are calculating the horizon line as the cross product of the first and second columns of p1, normalized by the Euclidean norm of the first two elements squared plus the second two elements squared.
    return (np.cross(p1[:, zero], p1[:, one]) / np.linalg.norm(p1[:two, zero] ** two + p1[:two, one] ** two) ** pointFive)

# Here we are creating a function called plot_horizon_line.
def plot_horizon_line(p1, p2):
    """
    Plots the horizon line.
    """

    plt.figure()   # Here we are creating a new figure for plotting.

    plt.imshow(p1)   # Here we are displaying the input image 'p1'.

    # Here we are plotting a yellow line representing the horizon using the input parameters 'p2'.
    plt.plot((np.arange((p1.shape[one]))), ((-p2[two] - p2[zero] * (np.arange((p1.shape[one])))) / p2[one]), 'b', linewidth = two)

    plt.show()    # Here we are showing the plotted image with the horizon line.

In [44]:
def get_camera_parameters():
    """
    Computes the camera parameters. Hint: The SymPy package is suitable for this.
    """
    # <YOUR IMPLEMENTATION>
    pass

def get_rotation_matrix():
    """
    Computes the rotation matrix using the camera parameters.
    """
    # <YOUR IMPLEMENTATION>
    pass

In [45]:
# Here we are creating a function called estimate_height.
def estimate_height(vanishingPoints, generatedHorizonline, personHeight, personCoordinates, gableCoordinates, cslPicture):
    """
    Estimates height for a specific object using the recorded coordinates. You might need to plot additional images here for
    your report.
    """

    _, subplotTwo = plt.subplots()   # Here we are creating a subplot for plotting.

    subplotTwo.imshow(cslPicture, aspect = 'equal')   # Here we are displaying the input image on the subplot.

    # Here we are plotting a blue line to represent the horizon line in the CSL image (part 1).
    subplotTwo.plot([(vanishingPoints[:, zero].reshape(three, one))[zero], (vanishingPoints[:, one].reshape(three, one))[zero]],

                    # Here we are plotting a blue line to represent the horizon line in the CSL image (part 2).
                    [(vanishingPoints[:, zero].reshape(three, one))[one], (vanishingPoints[:, one].reshape(three, one))[one]], 'b', linewidth = one)

    # Here we are plotting a red line to represent the person height in the CSL image.
    subplotTwo.plot([personCoordinates[zero][zero], personCoordinates[zero][one]], [personCoordinates[one][zero], personCoordinates[one][one]], 'r', linewidth = one)

    # Here we are plotting a green line to represent the gable height in the CSL image.
    subplotTwo.plot([gableCoordinates[zero][zero], gableCoordinates[zero][one]], [gableCoordinates[one][zero], gableCoordinates[one][one]], 'g', linewidth = one)

    # Here we are calculating the cross product of the real part of the cross product of the person's 2D coordinates extended to 3D with the gable's extended 3D coordinates.
    partOne = ((np.real(np.cross((np.real(np.cross((np.array([personCoordinates[zero][zero], personCoordinates[one][zero], one])).T,

              # Here we are calculating the cross product of the gable's 2D coordinates extended to 3D with the generated horizon line.
              (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one])).T))), generatedHorizonline))) /

              # Here we are calculating the real part of the cross product of the previously calculated cross product with the generated horizon line.
              (np.real(np.cross((np.real(np.cross((np.array([personCoordinates[zero][zero], personCoordinates[one][zero], one])).T,

              # Here we are repeating the cross product calculation with the gable's coordinates and the generated horizon line.
              (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one])).T))), generatedHorizonline)))[two])

    # Here we are computing the cross product of the transpose of partOne and a vector derived from personCoordinates and computing the cross product of the result above and another vector derived from gableCoordinates.
    partTwo = ((np.real(np.cross((np.real(np.cross(partOne.T, (np.array([personCoordinates[zero][one], personCoordinates[one][one], one])).T))).T,

              # Here we are computing the cross product of two vectors derived from gableCoordinates.
              (np.real(np.cross((np.array([gableCoordinates[zero][one], gableCoordinates[one][one], one])).T,

              # Here we are dividing the above result by the real part of the cross product of the following two vectors.
              (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one])).T))).T))) / (np.real(np.cross

              # Here we are computing the cross product of the transpose of partOne and a vector derived from personCoordinates.
              ((np.real(np.cross(partOne.T, (np.array([personCoordinates[zero][one], personCoordinates[one][one], one])).T))).T,

              # Here we are computing the cross product of two vectors derived from gableCoordinates.
              (np.real(np.cross((np.array([gableCoordinates[zero][one], gableCoordinates[one][one], one])).T,

              # Here we are accessing the element at index 'two' from the final computed array.
              (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one])).T))).T)))[two])

    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 1).
    subplotTwo.plot([partOne[zero], (np.array([personCoordinates[zero][zero], personCoordinates[one][zero], one]))[zero]],

                    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 2).
                    [partOne[one], (np.array([personCoordinates[zero][zero], personCoordinates[one][zero], one]))[one]], 'y', linewidth = one)

    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 3).
    subplotTwo.plot([partOne[zero], (np.array([personCoordinates[zero][one], personCoordinates[one][one], one]))[zero]],

                    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 4).
                    [partOne[one], (np.array([personCoordinates[zero][one], personCoordinates[one][one], one]))[one]], 'y', linewidth = one)

    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 5).
    subplotTwo.plot([partOne[zero], partTwo[zero]], [partOne[one], partTwo[one]], 'y', linewidth = one)

    # Here we are plotting a red line to represent the person height over the CSL building in the CSL image (part 1).
    subplotTwo.plot([(np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))[zero], partTwo[zero]],

                    # Here we are plotting a red line to represent the person height over the CSL building in the CSL image (part 2).
                    [(np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))[one], partTwo[one]], 'r', linewidth = one)

    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 6).
    subplotTwo.plot([(np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))[zero], partOne[zero]],

                    # Here we are plotting a yellow line to represent how we are generating the gable heights using the person height as a reference (part 7).
                    [(np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))[one], partOne[one]], 'y', linewidth = one)

    plt.show()   # Here we are displaying the subplot with all the plotted lines.

    # Here we are also calculating the norm (distance) between a transformed gable coordinate and the vanishing point.
    return ((personHeight * (np.linalg.norm((np.array([gableCoordinates[zero][one], gableCoordinates[one][one], one])) -

           # Here we are calculating the norm (distance) between another transformed gable coordinate and the vanishing point.
           (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))) * np.linalg.norm((vanishingPoints[:, two].reshape(three, one)).T - partTwo))) /

           # Here we are calculating the denominator as the product of norms (distances) between two sets of gable coordinates and the vanishing point.
           (np.linalg.norm(partTwo - (np.array([gableCoordinates[zero][zero], gableCoordinates[one][zero], one]))) *

           # Here we are calculating the norm (distance) between the transformed vanishing point and a gable coordinate.
           np.linalg.norm((vanishingPoints[:, two].reshape(three, one)).T - (np.array([gableCoordinates[zero][one], gableCoordinates[one][one], one])))))


In [46]:
# Main function

In [47]:
im = np.asarray(Image.open('CSL.jpeg'))

In [48]:
# Part 1
# Get vanishing points for each of the directions
num_vpts = 3
vpts = np.zeros((3, num_vpts))
for i in range(num_vpts):
    print('Getting vanishing point %d' % i)
    # Get at least three lines from user input
    n, lines, centers = get_input_lines(im)

    # <YOUR IMPLEMENTATION> Solve for vanishing point

    # Here we are calculating and storing vanishing point for a given set of lines.
    vpts[:, i] = get_vanishing_point(lines)

    print("Pixel coordinates of the vanishing point: ", vpts[:, i])   # Here we are printing the coordinates of the vanishing point.

    # Plot the lines and the vanishing point
    plot_lines_and_vp(im, lines, vpts[:, i])

# <YOUR IMPLEMENTATION> Get the ground horizon line

horizon_line = get_horizon_line(vpts)   # Here we are computing the horizon line using vanishing points.

# Here we are printing the computed ground horizon line.
print("Pixel coordinates of the ground horizon line:", horizon_line)

# <YOUR IMPLEMENTATION> Plot the ground horizon line

plot_horizon_line(im, horizon_line)   # Here we are plotting and visualize the horizon line on an image.

plt.savefig('ground_horizon_line.png')   # Here we are saving the plotted ground horizon line as an image.


Getting vanishing point 0
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Pixel coordinates of the vanishing point:  [-237.03344365  215.69639336    1.        ]
Getting vanishing point 1
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Click the two endpoints, use the right key to undo, and use the middle key to stop input
Pixel coordinates of the vanishing point:  [1.3565499

In [49]:
# # Part 2
# # <YOUR IMPLEMENTATION> Solve for the camera parameters (f, u, v)
# f, u, v = get_camera_parameters()
#
# # Part 3
# # <YOUR IMPLEMENTATION> Solve for the rotation matrix
# R = get_rotation_matrix()

In [53]:
# Part 4
# Record image coordinates for each object and store in map

# Here we are defining a tuple called objects containing the values: person, Gable 1, and Gable 2.
objects = ('person', 'Gable 1', 'Gable 2')

coords = dict()
for obj in objects:
    coords[obj] = get_top_and_bottom_coordinates(im, obj)

# Here we are printing the values of the coords variable.
print(coords)

# <YOUR IMPLEMENTATION> Estimate heights
for obj in objects[1:]:
    print('Estimating height of %s' % obj)

    # Here we are calculating the gable heights by calling the estimate_height function with parameters.
    height = estimate_height(vpts, horizon_line, inchesNumber, coords['person'], coords[obj], im)

    print(height)   # Here we are printing the calculated gable heights.


Click on the top coordinate of person
Click on the bottom coordinate of person
Click on the top coordinate of Gable 1
Click on the bottom coordinate of Gable 1
Click on the top coordinate of Gable 2
Click on the bottom coordinate of Gable 2
{'person': array([[628.0378072 , 625.73012785],
       [468.51275119, 511.20481918],
       [  1.        ,   1.        ]]), 'Gable 1': array([[506.8846413 , 505.73080162],
       [ 98.13021544, 146.5914818 ],
       [  1.        ,   1.        ]]), 'Gable 2': array([[900.34397056, 899.19013088],
       [102.74557414, 145.43764212],
       [  1.        ,   1.        ]])}
Estimating height of Gable 1
134.12733627153062
Estimating height of Gable 2
136.85295727113828


objc[4608]: Class FIFinderSyncExtensionHost is implemented in both /System/Library/PrivateFrameworks/FinderKit.framework/Versions/A/FinderKit (0x7fffa076ecd0) and /System/Library/PrivateFrameworks/FileProvider.framework/OverrideBundles/FinderSyncCollaborationFileProviderOverride.bundle/Contents/MacOS/FinderSyncCollaborationFileProviderOverride (0x11847fcd8). One of the two will be used. Which one is undefined.
